# import dataset

In [ ]:
import kagglehub

# Download the pre-packaged PhysioNet 2019 Sepsis Challenge files from Kaggle
path = kagglehub.dataset_download("salikhussaini49/prediction-of-sepsis")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd

csv_file_path = os.path.join(path, "Dataset.csv")
df = pd.read_csv(csv_file_path)

print(f"Dataset shape: {df.shape}")

# data cleaning

In [ ]:

id_col = 'Patient_ID' # Make sure this matches your exact column name

# Group by patient, forward-fill, grab the last row, and KEEP the index as a column
df_collapsed = df.groupby(id_col, group_keys=False).apply(lambda group: group.ffill().iloc[-1]).reset_index()

# Get the true sepsis max status for each patient
sepsis_status = df.groupby(id_col)['SepsisLabel'].max().reset_index()

# Drop the old label column and merge safely
df_collapsed = df_collapsed.drop(columns=['SepsisLabel'], errors='ignore').merge(sepsis_status, on=id_col)

print(f"Collapsed shape (one row per patient): {df_collapsed.shape}")
print(f"Positive sepsis cases remaining: {df_collapsed['SepsisLabel'].sum()}")

In [ ]:
df_collapsed.info()

In [ ]:
# Drop the unneeded columns
columns_to_drop = ['Unnamed: 0', 'Hour', 'Unit1', 'Unit2', 'HospAdmTime', 'Patient_ID', 'ICULOS']
df_cleaned = df_collapsed.drop(columns=columns_to_drop, errors='ignore')

# Separate Features (X) and Target (y)
# Target is SepsisLabel (column 42)
X_cleaned = df_cleaned.drop(columns=['SepsisLabel'])
y_cleaned = df_cleaned['SepsisLabel']

print(f"Features shape: {X_cleaned.shape}")
print(f"Target shape: {y_cleaned.shape}")


# train baseline model

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, roc_curve, precision_recall_curve
from sklearn.model_selection import GridSearchCV, StratifiedKFold
import matplotlib.pyplot as plt

X_train, X_test, y_train, y_test = train_test_split(
    X_cleaned, y_cleaned, test_size=0.2, random_state=42, stratify=y_cleaned
)

neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_weight = neg_count / pos_count

from xgboost import XGBClassifier

# Initialize the base model with your fixed weights & constants
base_model = XGBClassifier(
    scale_pos_weight=scale_weight,
    random_state=42,
    eval_metric='logloss'
)

# Define a hyperparameter grid around the baseline configuration
param_grid = {
    'max_depth': [4, 5, 6],
    'learning_rate': [0.01, 0.03, 0.05],
    'n_estimators': [150, 200, 300],
    'subsample': [0.6, 0.8],
    'colsample_bytree': [0.6, 0.8]
}

# Use StratifiedKFold to preserve thesepsis positive ratio across cross-validation splits
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Set up GridSearchCV optimizing for PR-AUC (average_precision)
grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    scoring='average_precision', 
    cv=cv,
    verbose=1,
    n_jobs=-1  # Uses all available CPU cores to speed up training
)

# Fit the grid search on the training data
grid_search.fit(X_train, y_train)

# Output the winning combination
print("\n--- Best Hyperparameters Found ---")
for param, value in grid_search.best_params_.items():
    print(f"{param}: {value}")

print(f"Best Cross-Validation PR-AUC: {grid_search.best_score_:.4f}")

# Extract optimized model ready for evaluation
best_model = grid_search.best_estimator_

# Evaluate Probabilities
y_pred_proba_tuned = best_model.predict_proba(X_test)[:, 1]

positive_indices = np.where(y_test == 1)[0]
positive_probs = y_pred_proba_tuned[positive_indices]

print(f"--- Tuned Probabilities for True Sepsis Cases ---")
print(f"Max probability assigned to a true sepsis case: {positive_probs.max():.4f}")
print(f"Mean probability assigned to a true sepsis case: {positive_probs.mean():.4f}\n")

# Find Optimal Threshold via PR Curve
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba_tuned)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

y_pred_tuned_thresh = (y_pred_proba_tuned >= best_threshold).astype(int)

# Print Metrics
print(f"Optimal Threshold: {best_threshold:.5f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba_tuned):.4f}")
print(f"PR-AUC: {average_precision_score(y_test, y_pred_proba_tuned):.4f}\n")
print("--- Tuned Classification Report ---")
print(classification_report(y_test, y_pred_tuned_thresh, zero_division=0))

# SHAP analysis

In [ ]:
import shap

X_train, X_test, y_train, y_test = train_test_split(
    X_cleaned, y_cleaned, test_size=0.2, random_state=42, stratify=y_cleaned
)


# Initialize the Tree Explainer on our best-performing tuned model
explainer = shap.TreeExplainer(best_model)

# Calculate SHAP values for the test set
shap_values = explainer(X_test)

# Global Feature Importance Bar Plot
# This displays the mean absolute SHAP value for each feature across the dataset
print("--- Global Feature Importance (Bar Plot) ---")
shap.summary_plot(shap_values, X_test, plot_type="bar")

# Detailed Bee Swarm Summary Plot
# This shows both feature importance AND the direction of impact (high vs. low feature values)
print("--- SHAP Bee Swarm Plot (Impact & Direction) ---")
shap.summary_plot(shap_values, X_test)

# add delta columns

In [ ]:
import pandas as pd
import numpy as np

exclude_cols = {'Unnamed: 0', 'Hour', 'Unit1', 'Unit2', 'HospAdmTime', 'ICULOS', 'SepsisLabel', 'Age', 'Gender'}
feature_cols = [col for col in df.columns if col not in exclude_cols]

print("Calculating vectorized delta features...")

# Compute non-null counts, first values, and last values across all groups 
counts = df.groupby(id_col)[feature_cols].count()
firsts = df.groupby(id_col)[feature_cols].first() # Pandas first() ignores NaNs by default
lasts = df.groupby(id_col)[feature_cols].last()   # Pandas last() ignores NaNs by default

# Vectorized subtraction: last minus first
deltas = lasts - firsts

# if there are fewer than 2 measurements, set to NaN
deltas[counts < 2] = np.nan

# Rename columns with the '_delta' suffix and reset index to bring back the ID column
deltas = deltas.rename(columns={col: f"{col}_delta" for col in feature_cols}).reset_index()

# Merge the new delta features into your existing dataset
df_collapsed_deltas = df_collapsed.merge(deltas, on='Patient_ID')

columns_to_drop = ['Unnamed: 0', 'Hour', 'Unit1', 'Unit2', 'HospAdmTime', 'Patient_ID', 'ICULOS', 'Patient_ID_delta']
df_cleaned_deltas = df_collapsed_deltas.drop(columns=columns_to_drop, errors='ignore')

print(f"Updated collapsed shape with delta features: {df_cleaned_deltas.shape}")

In [ ]:
df_cleaned_deltas.info()

In [ ]:
df_cleaned_deltas.head(20)

In [ ]:
X_cleaned_deltas = df_cleaned_deltas.drop(columns=['SepsisLabel'])
y_cleaned_deltas = df_cleaned_deltas['SepsisLabel']

# train new model with deltas

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, roc_curve, precision_recall_curve
from sklearn.model_selection import GridSearchCV, StratifiedKFold
import matplotlib.pyplot as plt

X_train, X_test, y_train, y_test = train_test_split(
    X_cleaned_deltas, y_cleaned_deltas, test_size=0.2, random_state=42, stratify=y_cleaned
)

neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_weight = neg_count / pos_count

from xgboost import XGBClassifier

# Initialize the base model with your fixed weights & constants
base_model = XGBClassifier(
    scale_pos_weight=scale_weight,
    random_state=42,
    eval_metric='logloss'
)

# Define a hyperparameter grid
param_grid = {
    'max_depth': [4, 5, 6],
    'learning_rate': [0.01, 0.03, 0.05],
    'n_estimators': [150, 200, 300],
    'subsample': [0.6, 0.8],
    'colsample_bytree': [0.6, 0.8]
}


# Use StratifiedKFold to preserve the sepsis positive ratio across cross-validation splits
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Set up GridSearchCV optimizing for PR-AUC (average_precision)
grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    scoring='average_precision', 
    cv=cv,
    verbose=1,
    n_jobs=-1  # Uses all available CPU cores to speed up training
)

# Fit the grid search on training data
grid_search.fit(X_train, y_train)

# Output the winning combination
print("\n--- Best Hyperparameters Found ---")
for param, value in grid_search.best_params_.items():
    print(f"{param}: {value}")

print(f"Best Cross-Validation PR-AUC: {grid_search.best_score_:.4f}")

# Extract optimized model ready for evaluation
best_model = grid_search.best_estimator_

# Evaluate Probabilities
y_pred_proba_tuned = best_model.predict_proba(X_test)[:, 1]

positive_indices = np.where(y_test == 1)[0]
positive_probs = y_pred_proba_tuned[positive_indices]

print(f"--- Tuned Probabilities for True Sepsis Cases ---")
print(f"Max probability assigned to a true sepsis case: {positive_probs.max():.4f}")
print(f"Mean probability assigned to a true sepsis case: {positive_probs.mean():.4f}\n")

# Find Optimal Threshold via PR Curve
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba_tuned)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

y_pred_tuned_thresh = (y_pred_proba_tuned >= best_threshold).astype(int)

# Print Metrics
print(f"Optimal Threshold: {best_threshold:.5f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba_tuned):.4f}")
print(f"PR-AUC: {average_precision_score(y_test, y_pred_proba_tuned):.4f}\n")
print("--- Tuned Classification Report ---")
print(classification_report(y_test, y_pred_tuned_thresh, zero_division=0))

# SHAP analysis with deltas

In [ ]:
import shap

X_train, X_test, y_train, y_test = train_test_split(
    X_cleaned_deltas, y_cleaned_deltas, test_size=0.2, random_state=42, stratify=y_cleaned
)


# Initialize the Tree Explainer on our best-performing tuned model
explainer = shap.TreeExplainer(best_model)

# Calculate SHAP values for the test set
shap_values = explainer(X_test)

# Global Feature Importance Bar Plot
# This displays the mean absolute SHAP value for each feature across the dataset
print("--- Global Feature Importance (Bar Plot) ---")
shap.summary_plot(shap_values, X_test, plot_type="bar")

# Detailed Bee Swarm Summary Plot
# This shows both feature importance AND the direction of impact (high vs. low feature values)
print("--- SHAP Bee Swarm Plot (Impact & Direction) ---")
shap.summary_plot(shap_values, X_test)